# Text Classification With RNNs

In this lab, we'll use RNNs to find the author of a text.

The corpus comes from classic works of English-language literature: 9 authors, 2 books per author, with one file per chapter.

## Downloading the Data From a Git Repository

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s classical-authors

## Downloading the spacy Model for English

In [ ]:
spacy_updated = !pip freeze | grep spacy==3
if not spacy_updated:
  !pip install 'spacy >= 3, < 4'
!python -m spacy download en_core_web_sm

## Installing and Importing PyTorch Lightning and the Other Required Libraries

In [ ]:
!pip install -q lightning torchmetrics

In [ ]:
import collections
import collections.abc
import pathlib

import lightning
import matplotlib.pyplot as plt
import numpy
import seaborn
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import spacy
import torch
import torchmetrics
import tqdm.notebook
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.utilities.model_summary import ModelSummary
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset, random_split

## Preprocessing

- Each sentence is an example
- Every word (or group of words) that is a named entity is replaced by its type (with the first letter tripled to create new "words")

In [ ]:
nlp = spacy.load(
    "en_core_web_sm",
    exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"))
nlp.enable_pipe("senter")


def replace_ners(tokens: collections.abc.Iterable[spacy.tokens.Token]
                 ) -> collections.abc.Iterator[str]:
  for token in tokens:
    if token.ent_iob_ == "O":
      yield token.text
    elif token.ent_iob_ == "B":
      prefix = token.ent_type_[0]
      yield f"{prefix}{prefix}{token.ent_type_}"


def get_data(directory: pathlib.Path,
             ) -> tuple[list[str], list[str]]:
  texts = []
  authors = []
  paths = list(directory.glob("*/*/*.txt"))
  contents = (p.read_text(encoding="utf8").replace("\n", " ") for p in paths)
  for path, doc in tqdm.notebook.tqdm(zip(paths, nlp.pipe(contents,
                                                          n_process=1)),
                                      total=len(paths)):

    author = path.parent.parent.name
    for sentence in doc.sents:
      authors.append(author)
      texts.append(" ".join(replace_ners(sentence)))

  return texts, authors


texts, authors = get_data(pathlib.Path("classical-authors"))

label_encoder = sklearn.preprocessing.LabelEncoder()
y = torch.tensor(label_encoder.fit_transform(authors))

print("Number of texts and shape of y:", len(texts), tuple(y.shape))

X_train_raw, X_test_raw, y_train, y_test = \
    sklearn.model_selection.train_test_split(texts, y, test_size=0.3)

In [ ]:
print(len(X_train_raw), tuple(y_train.shape))

## Preparing the Data as Sequences

### Preparing the Dictionary and the Sequences

With the `Tokenizer` class defined in the next cell:
- Build a dictionary on the `X_train_raw` corpus
- What's the vocabulary size?
- What's the maximum length, in words, of a sentence?

In [ ]:
class Tokenizer:
  """Map each word of a corpus to an index, by decreasing frequency."""

  # Characters removed from the text (replaced with spaces) before splitting
  filters = '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'

  def __init__(self) -> None:
    self.table = str.maketrans(self.filters, " " * len(self.filters))
    self.word_index: dict[str, int] = {}

  def split(self, text: str) -> list[str]:
    return text.lower().translate(self.table).split()

  def fit_on_texts(self, texts: collections.abc.Iterable[str]) -> None:
    counts = collections.Counter(word
                                 for text in texts
                                 for word in self.split(text))
    # Index 0 is reserved for padding, so words are numbered starting
    # from 1
    self.word_index = {
        word: index
        for index, (word, _) in enumerate(counts.most_common(), start=1)}

  def texts_to_sequences(self,
                         texts: collections.abc.Iterable[str]
                         ) -> list[list[int]]:
    return [[self.word_index[word]
             for word in self.split(text)
             if word in self.word_index]
            for text in texts]

In [ ]:
# Your code here

#### Solution

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train_raw)

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(s.split()) for s in X_train_raw)
print(f"Vocabulary size: {vocab_size}")
print(f"Length of the longest word sequence: {max_length}")

### Building the Matrix of Index Sequences

With the `Tokenizer` class and the `pad_sequences` function defined in the next cell:
- Turn `X_train_raw` and `X_test_raw` into sequences of word indices in a dictionary (``texts_to_sequences()`` method of the ``Tokenizer`` object instantiated previously)
- **Pad** the sequences so that they have a reasonable length to be processed by a bidirectional GRU:
  - We'll use `maxlen = 150`, because beyond that, GRUs start failing to properly carry the information.
  - `pad_sequences` fills each sequence with 0s at its end, index 0 being reserved for padding by the `Tokenizer`.
- Store the resulting sequences in `X_train_pad` and `X_test_pad`

In [ ]:
def pad_sequences(sequences: collections.abc.Iterable[list[int]],
                  maxlen: int) -> torch.Tensor:
  """Truncate each sequence to its last maxlen indices, pad it at its end."""
  sequences = list(sequences)
  padded = torch.zeros(len(sequences), maxlen, dtype=torch.long)
  for i, sequence in enumerate(sequences):
    truncated = torch.tensor(sequence[-maxlen:], dtype=torch.long)
    padded[i, :len(truncated)] = truncated
  return padded

In [ ]:
# Your code here

#### Solution

In [ ]:
max_length = 150

X_train_tokens = tokenizer.texts_to_sequences(X_train_raw)
X_test_tokens = tokenizer.texts_to_sequences(X_test_raw)

X_train_pad = pad_sequences(X_train_tokens, maxlen=max_length)
X_test_pad = pad_sequences(X_test_tokens, maxlen=max_length)

In [ ]:
print(f"Shape of the document corpus:  {tuple(X_train_pad.shape)}")
print(f"First example:  {X_train_pad[0]}")

## Modeling

Build a model with the following characteristics:
  - An embedding layer of size 300, whose padding vector stays zero (`padding_idx=0`)
  - A layer of bidirectional [`GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)s (`bidirectional=True` option):
    - of size `64`
    - an orthogonal initialization of **the** weight matrices
    - a dropout of `0.2` applied to the embeddings, at the input of the recurrent layer
  - A dense neural network layer that takes as input the last hidden states of both directions of the [`GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)
  - The GRU must not read the padding (see the "Ignoring Padding in the RNN" slide): without care, the backward direction would start its reading with dozens of padding steps, and the forward direction's hidden state would be that of the last padding step. Give it the sequences packed with [`pack_padded_sequence`](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html), from their actual lengths
  - A **cross-entropy** based loss function
  - The `adam` optimizer
  - `accuracy` as the evaluation metric

In [ ]:
class TextClassifier(lightning.LightningModule):
  """Lightning wrapper: predict the author of a padded sequence of words."""

  def __init__(self,
               model: nn.Module,
               learning_rate: float = 1e-3,
               sequence_len: int = max_length,
               num_classes: int = len(label_encoder.classes_)) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # An example input lets Lightning display the shapes of the input and output
    # tensors of each layer in the model summary
    self.example_input_array = torch.zeros(1, sequence_len, dtype=torch.long)
    # torchmetrics needs one metric instance per stage
    self.accuracies = nn.ModuleDict({
        f"{stage}_accuracy": torchmetrics.Accuracy(task="multiclass",
                                                   num_classes=num_classes)
        for stage in ("train", "val", "test")
    })

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    return self.model(sequences)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    sequences, authors = batch
    logits = self(sequences)
    loss = nn.functional.cross_entropy(logits, authors)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, authors)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def test_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                batch_index: int) -> torch.Tensor:
    return self._step(batch, "test")

  def predict_step(self, batch: torch.Tensor | tuple[torch.Tensor, ...],
                   batch_index: int) -> torch.Tensor:
    sequences = batch[0] if isinstance(batch, (list, tuple)) else batch
    return self(sequences).softmax(dim=-1).cpu()

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate)

In [ ]:
# Your code here

### Solution

In [ ]:
EMBEDDING_DIM = 300


def orthogonal_(rnn: nn.RNNBase) -> nn.RNNBase:
  """Initialize the weight matrices of a recurrent layer orthogonally."""
  for name, parameter in rnn.named_parameters():
    if name.startswith("weight"):
      nn.init.orthogonal_(parameter)
  return rnn


class TextGRU(nn.Module):
  """Embed the words, summarize them with a GRU, score each author."""

  def __init__(self,
               embedding: nn.Embedding,
               hidden_size: int = 64,
               num_layers: int = 1,
               bidirectional: bool = True,
               dropout: float = 0.2,
               num_classes: int = 9) -> None:
    super().__init__()
    self.embedding = embedding
    # Dropout applied to the embeddings, at the input of the recurrent layer
    self.dropout = nn.Dropout(dropout)
    self.gru = orthogonal_(
        nn.GRU(embedding.embedding_dim,
               hidden_size,
               num_layers=num_layers,
               bidirectional=bidirectional,
               dropout=dropout if num_layers > 1 else 0.,
               batch_first=True))
    self.head = nn.Linear(hidden_size * (2 if bidirectional else 1),
                          num_classes)

  def forward(self, sequences: torch.Tensor) -> torch.Tensor:
    embedded = self.dropout(self.embedding(sequences))
    # Actual length of each sequence, without the padding (index 0): the GRU
    # skips those positions, and its last hidden states are those of the last
    # real words, in both directions
    lengths = (sequences != 0).sum(dim=1).clamp(min=1).cpu()
    packed = pack_padded_sequence(embedded, lengths, batch_first=True,
                                  enforce_sorted=False)
    _, hidden = self.gru(packed)
    if self.gru.bidirectional:
      # Concatenate the last hidden states of both directions
      summary = torch.cat([hidden[-2], hidden[-1]], dim=-1)
    else:
      summary = hidden[-1]
    return self.head(summary)


model = TextClassifier(
    TextGRU(nn.Embedding(vocab_size, EMBEDDING_DIM, padding_idx=0)))
print(ModelSummary(model, max_depth=-1))

## Training

Train your model on `X_train_pad`:
- with batches of size `1024`
- for `10` epochs
- using 30% of the training set for validation

In [ ]:
# Your code here

### Solution

In [ ]:
batch_size = 1024

train_dataset, val_dataset = random_split(
    TensorDataset(X_train_pad, y_train), [0.7, 0.3])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

trainer = lightning.Trainer(max_epochs=10,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="bidi_gru"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

## Evaluation

Evaluate your model's performance on the test set.

In [ ]:
# Your code here

### Solution

In [ ]:
test_loader = DataLoader(TensorDataset(X_test_pad, y_test),
                         batch_size=batch_size)
test_accuracy = trainer.test(model, test_loader)[0]["test_accuracy"]
print(f"Test set accuracy: {test_accuracy:.3f}")

## First Conclusions

*Given the results, what can we say about the quality of this training? Justify.*

Your answer here

### Solution

We observe overfitting: train $\approx$ 75% while validation and test $\approx$ 50%

## Using Pre-Trained Word Embeddings

To get better results, you'll use the pre-trained GloVe embeddings, rather than learning specific embeddings as before.

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

In [ ]:
def glove_path(embedding_dim: int) -> pathlib.Path:
  if embedding_dim in {50, 100, 200, 300}:
    return pathlib.Path("glove.6B.{}d.txt".format(embedding_dim))
  else:
    raise ValueError("embedding_dim must be in {50, 100, 200, 300}")

## Reusing the GloVe Word Embeddings

PyTorch's [`nn.Embedding`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html) layer can be initialized with a weight matrix where row i is the embedding of word i.

After a quick look at the `glove.6B.300d.txt` files:
- Create a layer `embedding_layer` of type [`nn.Embedding`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html):
  - Initialize it with the GloVe embeddings.
  - Initialize the words of our vocabulary that aren't in GloVe with the zero vector
  - Use the appropriate flag to prevent these embeddings from changing during training

In [ ]:
# Your code here

### Solution

In [ ]:

EMBEDDING_DIM = 300
embedding_matrix = numpy.zeros((vocab_size, EMBEDDING_DIM), dtype="float32")
found = 0
with glove_path(EMBEDDING_DIM).open() as fh:
  for line in fh:
    values = line.split(" ")
    word = values[0]
    if word in tokenizer.word_index:
      found += 1
      coeffs = numpy.array(values[1:], dtype="float32")
      embedding_matrix[tokenizer.word_index[word]] = coeffs

print(f"Using {found} pre-trained embeddings out of {vocab_size} words "
      "in the vocabulary")

embedding_layer = nn.Embedding.from_pretrained(
    torch.from_numpy(embedding_matrix), freeze=True, padding_idx=0)

## Modeling, Training and Evaluation

- Create a model identical to the one of the previous part, except that the embedding layer is the one we just instantiated from GloVe
- Train this model with the same parameters as in the previous part
- Evaluate it on the test data

NB: a deeper model, which does better than this first one, is proposed after the solution.

In [ ]:
# Your code here

### Solution

In [ ]:
model = TextClassifier(TextGRU(embedding_layer, dropout=0.2))
print(ModelSummary(model, max_depth=-1))

In [ ]:
trainer = lightning.Trainer(max_epochs=20,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="glove_bidi_gru"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

In [ ]:
trainer.test(model, test_loader)

In [ ]:
EMBEDDING_DIM = 300

model = TextClassifier(
    TextGRU(embedding_layer, bidirectional=False, dropout=0.))
print(ModelSummary(model, max_depth=-1))

In [ ]:
trainer = lightning.Trainer(max_epochs=10,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="glove_gru"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

## A Deeper Solution

In [ ]:
model = TextClassifier(
    TextGRU(embedding_layer, num_layers=2, dropout=0.5))
print(ModelSummary(model, max_depth=-1))

In [ ]:
trainer = lightning.Trainer(max_epochs=30,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="deep_bidi_gru"),
                            enable_checkpointing=False)
trainer.fit(model, train_loader, val_loader)

In [ ]:
trainer.test(model, test_loader)

## Displaying the Test Confusion Matrix

*Display the confusion matrix (using a heatmap is recommended).*

In [ ]:
# Your code here

### Solution

In [ ]:
y_pred_softmax = torch.cat(trainer.predict(model, test_loader))
y_pred = y_pred_softmax.argmax(dim=1)

In [ ]:
conf_mat = sklearn.metrics.confusion_matrix(y_test, y_pred)
seaborn.heatmap(conf_mat,
                annot=True,
                fmt="d",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                cmap="rocket_r")
plt.show()